# Imports

In [ ]:
import subprocess, sys, pathlib, os

project_root = pathlib.Path(os.getcwd())
while project_root != project_root.parent:
    if (project_root / ".git").exists():
        break
    project_root = project_root.parent

print("project_root:", project_root)

subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(project_root / "GraphReasoning")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(project_root / "HypothesisGeneration")], check=True)

In [ ]:
import HypothesisGeneration
print(HypothesisGeneration.__file__)

C:\dev\Hypothesis_generation_agent-Full\HypothesisGeneration\HypothesisGeneration\__init__.py


In [ ]:
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer, AutoModel
import networkx as nx
from pathlib import Path

In [ ]:
import sys

# Clear any stale cached imports
for _key in list(sys.modules.keys()):
    if 'HypothesisGeneration' in _key or 'GraphReasoning' in _key:
        del sys.modules[_key]

# Move editable install finders to the front of sys.meta_path so they
# run before Python's PathFinder (which picks up empty namespace packages)
try:
    from __editable___hypothesisgeneration_0_1_0_finder import _EditableFinder as _HGFinder
    from __editable___graphreasoning_0_2_0_finder import _EditableFinder as _GRFinder
    for _finder in [_HGFinder, _GRFinder]:
        if _finder in sys.meta_path:
            sys.meta_path.remove(_finder)
        sys.meta_path.insert(0, _finder)
    print("Editable finders moved to front of sys.meta_path")
except ImportError as e:
    print(f"Warning: {e}")

from HypothesisGeneration import (
    make_dir_if_needed,
    load_graph_with_text_as_JSON,
    return_giant_component_of_graph,
    load_embeddings,
    generate_node_embeddings,
    SciAgentsGroup,
)
from GraphReasoning import (
    save_embeddings,
)

In [ ]:
data_dir_source = Path("../inputs/initial_graph_biomed")
data_dir_output = Path("../outputs/initial_ouput_biomed")
make_dir_if_needed(data_dir_output)
make_dir_if_needed(data_dir_source)


'Directory already exists.'

## Tokenizer and Embedding

In [ ]:
tokenizer_model = "BAAI/bge-large-en-v1.5"
embedding_tokenizer = AutoTokenizer.from_pretrained(tokenizer_model)
embedding_model = AutoModel.from_pretrained(
    tokenizer_model,
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

## Graph

In [ ]:
graph_name = "prostate_cancer_kg_20260611_044745.graphml"
G = load_graph_with_text_as_JSON(data_dir=data_dir_source, graph_name=graph_name)
G = return_giant_component_of_graph(G)
G = nx.Graph(G)

## Embeddings


In [ ]:
embedding_name = "embeddings-prostate_cancer_kg_20260611_044745-bge_large_en_v1.5.pkl"
try:
    node_embeddings = load_embeddings(f"{data_dir_source}/{embedding_name}")
except:
    print("Node embeddings not loaded, need to regenerate.")
    node_embeddings = generate_node_embeddings(
        G,
        embedding_tokenizer,
        embedding_model,
    )
    print("Node embeddings regenerated. Saving them to avoid recomputation")
    save_embeddings(node_embeddings, f"{data_dir_source}/{embedding_name}")

##  Default LLM model per agent

- Path Generation Agent: gpt-5-nano
- Ontologist: gpt-5-mini
- Scientist: gpt-5 (reasoning)
- Hypothesis Expansion Agent: gpt-5-mini
- Novelty Critic Agent: gpt-5-mini
- Coding Tasks Agent: gpt-5

- Summary Agent: gpt-5-mini

In [ ]:
# API keys are loaded from ../.env by the dotenv cell below — no need to set them here.

In [ ]:
import os
from dotenv import load_dotenv

# Load API keys from .env file in project root
load_dotenv("../.env")

# Verify keys are loaded
for key in ["OPENAI_API_KEY", "SEMANTIC_SCHOLAR_API_KEY", "NCBI_API_KEY"]:
    if os.environ.get(key):
        print(f"{key}: loaded ({key[:4]}...)")
    else:
        print(f"WARNING: {key} not set! Please configure it in ../.env")

OPENAI_API_KEY: loaded (OPEN...)
SEMANTIC_SCHOLAR_API_KEY: loaded (SEMA...)
NCBI_API_KEY: loaded (NCBI...)


In [ ]:
from openai import OpenAI
from HypothesisGeneration.domains import DomainConfig
import os

PROSTATE_CANCER = DomainConfig(
    disease="prostate cancer",
    tcga_cohort="TCGA-PRAD",
    tcga_full_name="Prostate Adenocarcinoma (Firehose Legacy)",
    n_patients="~500",
    available_endpoints=["OS", "PFI", "BCR"],
    available_imaging_features=["Gleason grade", "TIL density", "Perineural invasion"],
    available_gene_sets={
        "AR signaling": ["AR", "KLK3", "TMPRSS2", "ERG"],
        "T-cell Exhaustion": ["PDCD1", "CTLA4", "LAG3"],
    },
    therapy_proxies=["Androgen deprivation therapy (ADT)", "AR inhibitors (enzalutamide, abiraterone)"],
    specialist_description="specializing in prostate cancer",
)

openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

sci_agents_group = SciAgentsGroup(
    G=G,
    embedding_tokenizer=embedding_tokenizer,
    embedding_model=embedding_model,
    node_embeddings=node_embeddings,
    llm_client=openai_client,
    use_llm_validation=True,
    domain=PROSTATE_CANCER,
)

In [ ]:
# Step 1: Clear all related module imports
import sys

# Remove all potentially stale imports
modules_to_remove = []
for module_name in sys.modules.keys():
    if 'GraphReasoning' in module_name or 'HypothesisGeneration' in module_name:
        modules_to_remove.append(module_name)

for module_name in modules_to_remove:
    del sys.modules[module_name]
    print(f"Removed: {module_name}")

print("\nCleared all old imports")
# Step 2: Re-import HypothesisGeneration.utils
import HypothesisGeneration.utils as utils

# Verify imports
print("Module location:", utils.__file__)
print("\nCheck if new functions exist:")
print("europepmc_search exists:", hasattr(utils, 'europepmc_search'))
print("biorxiv_search exists:", hasattr(utils, 'biorxiv_search'))
print("crossref_search exists:", hasattr(utils, 'crossref_search'))

Removed: HypothesisGeneration.domains
Removed: HypothesisGeneration.agents.coding_tasks_agent
Removed: HypothesisGeneration.agents.hypothesis_expansion_agent
Removed: GraphReasoning.graph_tools
Removed: GraphReasoning.utils
Removed: GraphReasoning.graph_generation
Removed: GraphReasoning.graph_analysis
Removed: GraphReasoning.openai_tools
Removed: GraphReasoning.agents
Removed: GraphReasoning
Removed: HypothesisGeneration.utils
Removed: HypothesisGeneration.agents.novelty_critic_agent
Removed: HypothesisGeneration.agents.multi_critic_coordinator
Removed: HypothesisGeneration.agents.multi_critic_agent
Removed: HypothesisGeneration.agents.ontologist
Removed: HypothesisGeneration.agents.path_generation_agent
Removed: HypothesisGeneration.modifiedAutogen.console
Removed: HypothesisGeneration.modifiedAutogen.openai_agent
Removed: HypothesisGeneration.modifiedAutogen
Removed: HypothesisGeneration.agents.scientist
Removed: HypothesisGeneration.agents.summary_agent
Removed: HypothesisGeneratio

In [ ]:
import asyncio

usage = await sci_agents_group.run_with_console(
    message="Find a prognostic biomarker for biochemical recurrence in prostate cancer.",
    save_to=data_dir_output / "example_run.txt",
)

print(f"Total tokens used: {usage['total_tokens']}")
print(f"Time: {usage['active_time']}")

In [ ]:
# Parse the saved example run into markdown using report_parser
from HypothesisGeneration.report_parser import parse_log_to_markdown

input_path = data_dir_output / "example_run.txt"
output_path = data_dir_output / "example_run.md"
if input_path.exists():
    log_text = input_path.read_text(encoding="utf-8")
    md = parse_log_to_markdown(log_text)
    output_path.write_text(md, encoding="utf-8")
    print(f"Wrote markdown to {output_path}")
else:
    print(f"Input file not found: {input_path}")

Wrote markdown to ..\outputs\initial_ouput_biomed\example_run.md


In [ ]:
Consensus: 0.608

In [ ]:
# import asyncio
# from collections import Counter
# from HypothesisGeneration.agents.multi_critic_coordinator import (
#     MultiCriticCoordinator,
#     create_multi_critic_coordinator
# )

# async def detailed_test(hypothesis: str, config: dict = None):
#     """Detailed test with full analysis."""
    
#     if config is None:
#         config = {
#             "base_model": "gpt-4o",
#             "literature_sources": ["pubmed", "europepmc", "semantic_scholar", "crossref", "biorxiv", "arxiv"]
#         }
    
#     print("\n" + "="*80)
#     print("DETAILED NOVELTY CRITIC TEST")
#     print("="*80)
    
#     print("\nConfiguration:")
#     print(f"  Model: {config['base_model']}")
#     print(f"  Literature Sources: {', '.join(config['literature_sources'])}")
    
#     print(f"\nHypothesis:")
#     print(f"  {hypothesis[:200]}..." if len(hypothesis) > 200 else f"  {hypothesis}")
    
#     coordinator_agent = create_multi_critic_coordinator(**config)
#     coordinator = coordinator_agent.coordinator
    
#     print("\n" + "="*80)
#     print("EVALUATION IN PROGRESS")
#     print("="*80)
    
#     result = await coordinator.evaluate(hypothesis)
    
#     print("\n" + "="*80)
#     print("DETAILED ANALYSIS")
#     print("="*80)
    
#     method = result.get('method', 'unknown')
#     review_process = result.get('review_process', {})
#     scores = review_process.get('individual_scores', [])
    
#     print(f"\n1. Overall Scoring:")
#     print(f"  Method: {method.upper()}")
#     print(f"  Individual Scores: {scores}")
#     print(f"  Mean: {review_process.get('mean', 0):.2f}")
#     print(f"  Std Dev: {review_process.get('std', 0):.2f}")
    
#     print(f"\n2. Individual Reviewer Analysis:")
    
#     individual_results = result.get('individual_results', [])
    
#     for i, reviewer_result in enumerate(individual_results, 1):
#         reviewer_role = reviewer_result.get('reviewer_role', f'Reviewer {i}')
#         print(f"\n  {'─'*70}")
#         print(f"  REVIEWER {i}: {reviewer_role.upper()}")
#         print(f"  {'─'*70}")
        
#         lit_summary = reviewer_result.get('literature_search_summary', {})
#         print(f"  Literature Search:")
#         print(f"     Searches performed: {lit_summary.get('total_searches_performed', 'N/A')}")
#         print(f"     Papers found: {lit_summary.get('total_relevant_papers_found', 'N/A')}")
        
#         queries = lit_summary.get('key_queries_used', [])
#         if queries:
#             print(f"     Key queries:")
#             for q in queries[:3]:
#                 print(f"       - {q}")
        
#         sources_used = lit_summary.get('search_sources_used', [])
#         if sources_used:
#             print(f"     Sources: {', '.join(sources_used)}")
        
#         novelty_breakdown = reviewer_result.get('novelty_breakdown', {})
#         print(f"\n  Novelty Breakdown:")
#         for dimension in ['technical_novelty', 'application_novelty', 'theoretical_novelty']:
#             dim_data = novelty_breakdown.get(dimension, {})
#             if dim_data:
#                 score = dim_data.get('score', 'N/A')
#                 dim_name = dimension.replace('_', ' ').title()
#                 print(f"     {dim_name}: {score}/10")
        
#         overall = reviewer_result.get('overall_assessment', {})
#         print(f"\n  Overall:")
#         print(f"     Score: {overall.get('overall_novelty_score', 'N/A')}/10")
#         print(f"     Recommendation: {reviewer_result.get('recommendation', 'N/A')}")
        
#         refs = reviewer_result.get('references', [])
#         print(f"     References: {len(refs)}")
#         if refs:
#             print(f"     Reference list:")
#             for idx, ref in enumerate(refs, 1):
#                 print(f"       [{idx}] {ref}")
    
#     print(f"\n3. Consolidated References Analysis:")
    
#     consolidated_refs = result.get('consolidated_references', {})
#     total_refs = consolidated_refs.get('total_unique_references', 0)
#     print(f"  Total unique references: {total_refs}")
    
#     refs_by_reviewer = consolidated_refs.get('references_by_reviewer', {})
#     if refs_by_reviewer:
#         print(f"  References by reviewer:")
#         for reviewer, refs in refs_by_reviewer.items():
#             print(f"    {reviewer}: {len(refs)} refs")
    
#     all_refs = consolidated_refs.get('all_references', [])
#     if all_refs:
#         print(f"\n  Reference sources breakdown:")
#         source_counts = Counter()
#         for ref in all_refs:
#             url = ref.lower()
#             if 'pubmed' in url:
#                 source_counts['PubMed'] += 1
#             elif 'europepmc' in url:
#                 source_counts['Europe PMC'] += 1
#             elif 'semanticscholar' in url:
#                 source_counts['Semantic Scholar'] += 1
#             elif 'arxiv' in url:
#                 source_counts['arXiv'] += 1
#             elif 'doi.org' in url:
#                 source_counts['DOI/CrossRef'] += 1
#             else:
#                 source_counts['Other'] += 1
        
#         for source, count in source_counts.most_common():
#             percentage = (count / total_refs * 100) if total_refs > 0 else 0
#             print(f"    {source}: {count} ({percentage:.1f}%)")
        
#         print(f"\n  All consolidated references (deduplicated):")
#         for idx, ref in enumerate(all_refs, 1):
#             print(f"    [{idx}] {ref}")
    
#     print(f"\n4. Final Consensus:")
    
#     final = result.get('final_assessment', {})
#     score = final.get('consensus_score', 0)
#     confidence = final.get('confidence', 0)
#     recommendation = final.get('recommendation', 'N/A')
    
#     print(f"  Consensus Score: {score}/10")
#     print(f"  Confidence: {confidence:.1%}")
#     print(f"  Recommendation: {recommendation}")
    
#     print(f"\n  Score visualization:")
#     bar_length = int(score * 5)
#     bar = "█" * bar_length + "░" * (50 - bar_length)
#     print(f"  {bar} {score}/10")
    
#     print("\n" + "="*80)
#     if score >= 7:
#         print("CONCLUSION: Hypothesis APPROVED")
#     elif score >= 5:
#         print("CONCLUSION: Hypothesis needs REVISION")
#     else:
#         print("CONCLUSION: Hypothesis NOT APPROVED")
#     print("="*80 + "\n")
    
#     return result


# # ====== Run the test (key call in this notebook) ======

# hypothesis = "Among male TCGA-BLCA patients, high ADRM!expression coupled with a stroma-rich histol-ogy (high stromal-to-tumor ratio) and low TImargin enrichment on H&E images predicts inferior overall survival and progression-free interval, particularly when ECM/matrisome tran-scripts (e.8., FNI, COLIAI, COL3Al) and Tcell exhaustion signatures are upregulated. ThisADRMI-stroma-immune imaging-genomic axiswill show stronger adverse prognostic impact inpatients with comorbid diabetes, consistent withdiabetes-associated microenvironmental remod-eling."
# # If your Jupyter supports top-level await (most likely it does):
# result = await detailed_test(hypothesis)


DETAILED NOVELTY CRITIC TEST

Configuration:
  Model: gpt-4o
  Literature Sources: pubmed, europepmc, semantic_scholar, crossref, biorxiv, arxiv

Hypothesis:
  Among male TCGA-BLCA patients, high ADRM!expression coupled with a stroma-rich histol-ogy (high stromal-to-tumor ratio) and low TImargin enrichment on H&E images predicts inferior overall survival and...

EVALUATION IN PROGRESS
Starting novelty evaluation (Prover -> Verifier -> Judge)...

Prover -> Verifier -> Judge
Hypothesis: Among male TCGA-BLCA patients, high ADRM!expression coupled with a stroma-rich histol-ogy (high stromal-to-tumor ratio) ...
Max rounds: 3  |  Min rounds before verdict: 2

ROUND 0

[Prover] arguing...
[Prover] (3712 chars)
----------------------------------------
### Novelty and Scientific Plausibility of the Hypothesis:

#### 1. Novelty:

**ADRM1 Expression in Bladder Cancer:**
Currently, the literature does not extensively cover the role of ADRM1 specifically in bladder cancer prognosis. The search res

In [ ]:
# import asyncio
# from collections import Counter
# from HypothesisGeneration.agents.multi_critic_coordinator import (
#     MultiCriticCoordinator,
#     create_multi_critic_coordinator
# )

# async def detailed_test(hypothesis: str, config: dict = None):
#     """Detailed test for a single hypothesis (original function, logic unchanged)."""
    
#     if config is None:
#         config = {
#             "base_model": "gpt-4o",
#             "literature_sources": ["pubmed", "europepmc", "semantic_scholar", "crossref", "biorxiv", "arxiv"]
#         }
    
#     print("\n" + "="*80)
#     print("DETAILED NOVELTY CRITIC TEST")
#     print("="*80)
    
#     print("\nConfiguration:")
#     print(f"  Model: {config['base_model']}")
#     print(f"  Literature Sources: {', '.join(config['literature_sources'])}")
    
#     print(f"\nHypothesis:")
#     print(f"  {hypothesis[:200]}..." if len(hypothesis) > 200 else f"  {hypothesis}")
    
#     coordinator_agent = create_multi_critic_coordinator(**config)
#     coordinator = coordinator_agent.coordinator
    
#     print("\n" + "="*80)
#     print("EVALUATION IN PROGRESS")
#     print("="*80)
    
#     result = await coordinator.evaluate(hypothesis)
    
#     print("\n" + "="*80)
#     print("DETAILED ANALYSIS")
#     print("="*80)
    
#     method = result.get('method', 'unknown')
#     review_process = result.get('review_process', {})
#     scores = review_process.get('individual_scores', [])
    
#     print(f"\n1. Overall Scoring:")
#     print(f"  Method: {method.upper()}")
#     print(f"  Individual Scores: {scores}")
#     print(f"  Mean: {review_process.get('mean', 0):.2f}")
#     print(f"  Std Dev: {review_process.get('std', 0):.2f}")
    
#     print(f"\n2. Individual Reviewer Analysis:")
    
#     individual_results = result.get('individual_results', [])
    
#     for i, reviewer_result in enumerate(individual_results, 1):
#         reviewer_role = reviewer_result.get('reviewer_role', f'Reviewer {i}')
#         print(f"\n  {'─'*70}")
#         print(f"  REVIEWER {i}: {reviewer_role.upper()}")
#         print(f"  {'─'*70}")
        
#         lit_summary = reviewer_result.get('literature_search_summary', {})
#         print(f"  Literature Search:")
#         print(f"     Searches performed: {lit_summary.get('total_searches_performed', 'N/A')}")
#         print(f"     Papers found: {lit_summary.get('total_relevant_papers_found', 'N/A')}")
        
#         queries = lit_summary.get('key_queries_used', [])
#         if queries:
#             print(f"     Key queries:")
#             for q in queries[:3]:
#                 print(f"       - {q}")
        
#         sources_used = lit_summary.get('search_sources_used', [])
#         if sources_used:
#             print(f"     Sources: {', '.join(sources_used)}")
        
#         novelty_breakdown = reviewer_result.get('novelty_breakdown', {})
#         print(f"\n  Novelty Breakdown:")
#         for dimension in ['technical_novelty', 'application_novelty', 'theoretical_novelty']:
#             dim_data = novelty_breakdown.get(dimension, {})
#             if dim_data:
#                 score = dim_data.get('score', 'N/A')
#                 dim_name = dimension.replace('_', ' ').title()
#                 print(f"     {dim_name}: {score}/10")
        
#         overall = reviewer_result.get('overall_assessment', {})
#         print(f"\n  Overall:")
#         print(f"     Score: {overall.get('overall_novelty_score', 'N/A')}/10")
#         print(f"     Recommendation: {reviewer_result.get('recommendation', 'N/A')}")
        
#         refs = reviewer_result.get('references', [])
#         print(f"     References: {len(refs)}")
#         if refs:
#             print(f"     Reference list:")
#             for idx, ref in enumerate(refs, 1):
#                 print(f"       [{idx}] {ref}")
    
#     print(f"\n3. Consolidated References Analysis:")
    
#     consolidated_refs = result.get('consolidated_references', {})
#     total_refs = consolidated_refs.get('total_unique_references', 0)
#     print(f"  Total unique references: {total_refs}")
    
#     refs_by_reviewer = consolidated_refs.get('references_by_reviewer', {})
#     if refs_by_reviewer:
#         print(f"  References by reviewer:")
#         for reviewer, refs in refs_by_reviewer.items():
#             print(f"    {reviewer}: {len(refs)} refs")
    
#     all_refs = consolidated_refs.get('all_references', [])
#     if all_refs:
#         print(f"\n  Reference sources breakdown:")
#         source_counts = Counter()
#         for ref in all_refs:
#             url = ref.lower()
#             if 'pubmed' in url:
#                 source_counts['PubMed'] += 1
#             elif 'europepmc' in url:
#                 source_counts['Europe PMC'] += 1
#             elif 'semanticscholar' in url:
#                 source_counts['Semantic Scholar'] += 1
#             elif 'arxiv' in url:
#                 source_counts['arXiv'] += 1
#             elif 'doi.org' in url:
#                 source_counts['DOI/CrossRef'] += 1
#             else:
#                 source_counts['Other'] += 1
        
#         for source, count in source_counts.most_common():
#             percentage = (count / total_refs * 100) if total_refs > 0 else 0
#             print(f"    {source}: {count} ({percentage:.1f}%)")
        
#         print(f"\n  All consolidated references (deduplicated):")
#         for idx, ref in enumerate(all_refs, 1):
#             print(f"    [{idx}] {ref}")
    
#     print(f"\n4. Final Consensus:")
    
#     final = result.get('final_assessment', {})
#     score = final.get('consensus_score', 0)
#     confidence = final.get('confidence', 0)
#     recommendation = final.get('recommendation', 'N/A')
    
#     print(f"  Consensus Score: {score}/10")
#     print(f"  Confidence: {confidence:.1%}")
#     print(f"  Recommendation: {recommendation}")
    
#     print(f"\n  Score visualization:")
#     bar_length = int(score * 5)
#     bar = "█" * bar_length + "░" * (50 - bar_length)
#     print(f"  {bar} {score}/10")
    
#     print("\n" + "="*80)
#     if score >= 7:
#         print("CONCLUSION: Hypothesis APPROVED")
#     elif score >= 5:
#         print("CONCLUSION: Hypothesis needs REVISION")
#     else:
#         print("CONCLUSION: Hypothesis NOT APPROVED")
#     print("="*80 + "\n")
    
#     return result


# # ========= Batch testing for multiple hypotheses =========

# async def batch_test(hypotheses, config: dict = None):
#     """
#     Batch test multiple hypotheses.
#     hypotheses: list[str]
#     Returns: list[dict], each element is {"hypothesis": ..., "result": ...}
#     """
#     results = []
#     total = len(hypotheses)
    
#     for i, h in enumerate(hypotheses, 1):
#         print("\n" + "#"*80)
#         print(f"BATCH TEST {i}/{total}")
#         print("#"*80)
#         res = await detailed_test(h, config=config)
#         results.append({
#             "index": i,
#             "hypothesis": h,
#             "result": res
#         })
    
#     print("\n" + "#"*80)
#     print("ALL BATCH TESTS COMPLETED")
#     print("#"*80 + "\n")
#     return results

# batch_results = await batch_test(hypotheses)

NameError: name 'hypotheses' is not defined

In [ ]:
# Parse the saved example run into markdown using report_parser
from HypothesisGeneration.report_parser import parse_log_to_markdown

input_path = data_dir_output / "example_run.txt"
output_path = data_dir_output / "example_run.md"
if input_path.exists():
    log_text = input_path.read_text(encoding="utf-8")
    md = parse_log_to_markdown(log_text)
    output_path.write_text(md, encoding="utf-8")
    print(f"Wrote markdown to {output_path}")
else:
    print(f"Input file not found: {input_path}")

Wrote markdown to ..\outputs\initial_ouput_biomed\example_run.md
